# 🚀 創新多市場交易策略：打敗 Buy & Hold
## 20年回測分析 (2005-2025) - 台股與美股主流標的

### 策略特色
- **創新技術指標**：VAM (波動率調整動量)、MFS (資金流強度)、HMA (Hull移動平均)
- **多因子融合**：趨勢確認 + 動能過濾 + 資金流確認 + 波動率調整持倉
- **市場體制識別**：動態調整進出場條件
- **多市場測試**：美股 (SPY, QQQ, NVDA, AAPL, MSFT) + 台股 (0050.TW, 2330.TW, 2454.TW)
- **完整風險管理**：ATR動態停損停利 + 部位大小控制

### 作者資訊
- 建立日期：2025年
- 回測期間：2005/01/01 - 2025/01/01
- 資料來源：yfinance
- 策略類型：趨勢追蹤 + 動能 + 波動率調整

## 📋 目錄
1. [環境設定與函式庫匯入](#section1)
2. [創新技術指標實作](#section2)
3. [資料下載與預處理](#section3)
4. [交易策略邏輯](#section4)
5. [回測引擎實作](#section5)
6. [績效分析與視覺化](#section6)
7. [多標的比較分析](#section7)
8. [牛熊市分層分析](#section8)
9. [參數敏感度測試](#section9)
10. [結論與未來改進](#section10)
11. [生成式AI應用反思](#section11)

<a id='section1'></a>
## 1️⃣ 環境設定與函式庫匯入

In [ ]:
# 安裝必要函式庫（首次執行時需要）
# !pip install yfinance pandas numpy matplotlib plotly scipy statsmodels ta-lib

import warnings
warnings.filterwarnings('ignore')

# 基礎函式庫
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import yfinance as yf

# 視覺化
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# 統計分析
from scipy import stats
import statsmodels.api as sm

# 設定繪圖風格
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['font.size'] = 12

print("✅ 函式庫載入完成")
print(f"📅 當前時間：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

<a id='section2'></a>
## 2️⃣ 創新技術指標實作

### 2.1 VAM (Volatility-Adjusted Momentum) - 波動率調整動量指標
**公式**：`VAM = (收盤價 / N日前收盤價 - 1) / ATR(N) * 100`

**說明**：傳統動量指標未考慮波動率，本指標將動能除以ATR進行標準化，使不同波動率的標的可以公平比較。

### 2.2 MFS (Money Flow Strength) - 資金流強度指標
**公式**：`MFS = EMA(典型價格變化量 × 成交量, 14) / SMA(成交量, 14)`

**說明**：結合價格變化與成交量，衡量資金流入流出強度，領先於純價格指標。

### 2.3 HMA (Hull Moving Average) - Hull移動平均
**公式**：`HMA = WMA(2×WMA(n/2) - WMA(n), sqrt(n))`

**說明**：相比傳統MA，HMA延遲更低且更平滑，能更早捕捉趨勢反轉。

In [ ]:
def calculate_vam(df, period=14, atr_period=14):
    """
    計算波動率調整動量指標 (VAM)
    
    參數:
        df: 包含Close, High, Low的DataFrame
        period: 動量計算期間
        atr_period: ATR計算期間
    
    返回:
        新增VAM欄位的DataFrame
    """
    df = df.copy()
    
    # 計算動量
    df['momentum'] = df['Close'] / df['Close'].shift(period) - 1
    
    # 計算ATR
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = ranges.max(axis=1)
    df['atr'] = true_range.rolling(atr_period).mean()
    
    # 計算VAM
    df['vam'] = df['momentum'] / df['atr'] * 100
    
    # 清理臨時欄位
    df.drop(['momentum', 'atr'], axis=1, inplace=True)
    
    return df


def calculate_mfs(df, period=14):
    """
    計算資金流強度指標 (MFS)
    
    參數:
        df: 包含Close, Volume的DataFrame
        period: EMA/SMA計算期間
    
    返回:
        新增MFS欄位的DataFrame
    """
    df = df.copy()
    
    # 計算典型價格
    typical_price = (df['High'] + df['Low'] + df['Close']) / 3
    
    # 計算典型價格變化量
    price_change = typical_price.diff()
    
    # 計算資金流
    money_flow = price_change * df['Volume']
    
    # 計算MFS
    df['mfs_numerator'] = money_flow.ewm(span=period, adjust=False).mean()
    df['mfs_denominator'] = df['Volume'].rolling(period).mean()
    df['mfs'] = df['mfs_numerator'] / df['mfs_denominator'] * 1000
    
    # 清理臨時欄位
    df.drop(['mfs_numerator', 'mfs_denominator'], axis=1, inplace=True)
    
    return df


def calculate_hma(df, period=21):
    """
    計算Hull移動平均 (HMA)
    
    參數:
        df: 包含Close的DataFrame
        period: HMA計算期間
    
    返回:
        新增HMA欄位的DataFrame
    """
    df = df.copy()
    n = period
    half_n = int(n / 2)
    sqrt_n = int(np.sqrt(n))
    
    # 計算WMA
    def wma(series, window):
        weights = np.arange(1, window + 1)
        return series.rolling(window).apply(
            lambda prices: np.dot(prices, weights) / weights.sum(),
            raw=True
        )
    
    # HMA = WMA(2*WMA(n/2) - WMA(n), sqrt(n))
    wma_half = wma(df['Close'], half_n)
    wma_full = wma(df['Close'], n)
    df['hma'] = wma(2 * wma_half - wma_full, sqrt_n)
    
    return df


def calculate_all_indicators(df):
    """
    計算所有技術指標
    
    參數:
        df: 包含OHLCV的DataFrame
    
    返回:
        新增所有指標欄位的DataFrame
    """
    df = df.copy()
    
    # 基礎指標
    df['ema_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['ema_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['rsi_14'] = calculate_rsi(df['Close'], 14)
    df['macd'], df['macd_signal'], df['macd_hist'] = calculate_macd(df['Close'])
    
    # 創新指標
    df = calculate_vam(df, period=14, atr_period=14)
    df = calculate_mfs(df, period=14)
    df = calculate_hma(df, period=21)
    
    # 計算ATR用於停損停利
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = ranges.max(axis=1)
    df['atr'] = true_range.rolling(14).mean()
    
    return df


def calculate_rsi(prices, period=14):
    """計算RSI指標"""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))


def calculate_macd(prices, fast=12, slow=26, signal=9):
    """計算MACD指標"""
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram


print("✅ 技術指標函式定義完成")

<a id='section3'></a>
## 3️⃣ 資料下載與預處理

### 資料來源說明
- **美股**：使用 yfinance 從 Yahoo Finance 下載
- **台股**：使用 yfinance 從 Yahoo Finance 下載 (代碼需加上 .TW)

### 預處理步驟
1. 處理缺值：使用前向填補 (forward fill)
2. 日期格式轉換：確保索引為 datetime 類型
3. 調整後價格：使用 Adj Close 避免除權息影響
4. 交易量單位統一：台股乘以 1000 股

In [ ]:
def download_stock_data(ticker, start_date='2005-01-01', end_date='2025-01-01'):
    """
    下載股票歷史數據並進行預處理
    
    參數:
        ticker: 股票代碼 (如 'SPY', '2330.TW')
        start_date: 開始日期
        end_date: 結束日期
    
    返回:
        預處理後的 DataFrame
    """
    print(f"📥 下載 {ticker} 數據...")
    
    try:
        # 下載數據
        df = yf.download(ticker, start=start_date, end=end_date, progress=False)
        
        if df.empty:
            print(f"⚠️ 警告：{ticker} 無數據")
            return None
        
        # 處理多層級索引（新版 yfinance）
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        
        # 選擇必要欄位
        required_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
        available_cols = [col for col in required_cols if col in df.columns]
        df = df[available_cols].copy()
        
        # 處理缺值：使用前向填補
        df = df.ffill()
        df = df.bfill()  # 如果開頭有缺值，用後向填補
        
        # 確保索引為 datetime 類型
        df.index = pd.to_datetime(df.index)
        
        # 台股交易量單位轉換（如果有 .TW）
        if '.TW' in ticker and 'Volume' in df.columns:
            df['Volume'] = df['Volume'] * 1000
        
        # 移除全為 0 或 NaN 的列
        df = df.replace(0, np.nan).dropna()
        
        print(f"✅ {ticker} 下載完成：{len(df)} 筆數據 ({df.index[0].date()} ~ {df.index[-1].date()})")
        
        return df
    
    except Exception as e:
        print(f"❌ {ticker} 下載失敗：{str(e)}")
        return None


# 定義回測標的
tickers = {
    '美股': ['SPY', 'QQQ', 'NVDA', 'AAPL', 'MSFT'],
    '台股': ['0050.TW', '2330.TW', '2454.TW']
}

# 下載所有標的數據
print("="*60)
print("🚀 開始下載股票數據")
print("="*60)

all_data = {}
for market, ticker_list in tickers.items():
    print(f"\n📊 {market}市場:")
    for ticker in ticker_list:
        data = download_stock_data(ticker)
        if data is not None:
            all_data[ticker] = data

print(f"\n✅ 總共成功下載 {len(all_data)} 檔股票數據")

<a id='section4'></a>
## 4️⃣ 交易策略邏輯

### 創新策略：VAM-MFS-HMA 三因子趨勢追蹤系統

#### 進場條件 (需同時滿足以下所有條件)：
1. **趨勢確認**：收盤價 > HMA(21) AND HMA 斜率向上
2. **動能過濾**：VAM > 0.5 (正動能且經波動率調整)
3. **資金流確認**：MFS > 0 (資金淨流入)
4. **均線多頭排列**：EMA(20) > EMA(50)
5. **RSI 健康度**：45 < RSI(14) < 75 (避免超買超賣)

#### 出場條件 (滿足任一即出場)：
1. **停損**：收盤價 < 進場價 × (1 - 2.5×ATR%) 
2. **停利**：收盤價 < 最高價 × (1 - 1.5×ATR%) (移動停利)
3. **趨勢反轉**：收盤價 < HMA(21) AND HMA 斜率向下
4. **動能消失**：VAM < -0.3

#### 部位管理：
- **初始部位**：100% 資金
- **延遲進場**：訊號產生後隔日開盤進場
- **再進場限制**：出場後需等待 3 交易日才能再次進場

In [ ]:
def generate_trading_signals(df):
    """
    產生交易訊號
    
    參數:
        df: 包含所有技術指標的 DataFrame
    
    返回:
        新增訊號欄位的 DataFrame
    """
    df = df.copy()
    
    # 初始化訊號欄位
    df['signal'] = 0
    df['position'] = 0
    df['entry_price'] = np.nan
    df['exit_price'] = np.nan
    df['trade_return'] = np.nan
    
    # 計算 HMA 斜率
    df['hma_slope'] = df['hma'].diff()
    
    # 進場條件
    trend_condition = (df['Close'] > df['hma']) & (df['hma_slope'] > 0)
    momentum_condition = df['vam'] > 0.5
    moneyflow_condition = df['mfs'] > 0
    ma_condition = df['ema_20'] > df['ema_50']
    rsi_condition = (df['rsi_14'] > 45) & (df['rsi_14'] < 75)
    
    # 綜合進場訊號
    entry_signal = trend_condition & momentum_condition & moneyflow_condition & ma_condition & rsi_condition
    
    # 出場條件
    stop_loss = df['Close'] < df['entry_price'].ffill() * (1 - 2.5 * df['atr'] / df['Close'])
    trailing_stop = df['Close'] < (df['Close'].cummax() * (1 - 1.5 * df['atr'] / df['Close']))
    trend_reversal = (df['Close'] < df['hma']) & (df['hma_slope'] < 0)
    momentum_loss = df['vam'] < -0.3
    
    exit_signal = stop_loss | trailing_stop | trend_reversal | momentum_loss
    
    # 產生交易訊號 (考慮延遲進場和冷卻期)
    in_position = False
    cooldown = 0
    entry_price = 0
    highest_price = 0
    
    positions = []
    entry_prices = []
    exit_prices = []
    trade_returns = []
    
    for i in range(len(df)):
        if cooldown > 0:
            cooldown -= 1
        
        if not in_position:
            # 檢查進場訊號 (延遲一天進場，所以用前一天的訊號)
            if i > 0 and entry_signal.iloc[i-1] and cooldown == 0:
                in_position = True
                entry_price = df.iloc[i]['Open']  # 隔日開盤進場
                highest_price = entry_price
                positions.append(1)
                entry_prices.append(entry_price)
                exit_prices.append(np.nan)
                trade_returns.append(np.nan)
            else:
                positions.append(0)
                entry_prices.append(np.nan)
                exit_prices.append(np.nan)
                trade_returns.append(np.nan)
        else:
            # 更新最高價
            current_high = df.iloc[i]['High']
            if current_high > highest_price:
                highest_price = current_high
            
            # 檢查出場訊號
            current_price = df.iloc[i]['Close']
            current_atr = df.iloc[i]['atr']
            
            # 動態計算停損停利價位
            stop_loss_price = entry_price * (1 - 2.5 * current_atr / current_price)
            trailing_stop_price = highest_price * (1 - 1.5 * current_atr / current_price)
            
            if (current_price < stop_loss_price or 
                current_price < trailing_stop_price or 
                (exit_signal.iloc[i] and i > 0)):
                # 執行出場
                exit_price = df.iloc[i+1]['Open'] if i+1 < len(df) else current_price
                trade_return = (exit_price - entry_price) / entry_price
                
                positions.append(0)
                entry_prices.append(np.nan)
                exit_prices.append(exit_price)
                trade_returns.append(trade_return)
                
                in_position = False
                cooldown = 3  # 冷卻期 3 天
                entry_price = 0
                highest_price = 0
            else:
                positions.append(1)
                entry_prices.append(np.nan)
                exit_prices.append(np.nan)
                trade_returns.append(np.nan)
    
    df['position'] = positions
    df['entry_price'] = entry_prices
    df['exit_price'] = exit_prices
    df['trade_return'] = trade_returns
    
    return df


print("✅ 交易訊號產生函式定義完成")

<a id='section5'></a>
## 5️⃣ 回測引擎實作

In [ ]:
def run_backtest(df, initial_capital=100000):
    """
    執行回測
    
    參數:
        df: 包含訊號的 DataFrame
        initial_capital: 初始資金
    
    返回:
        回測結果 DataFrame 和績效統計
    """
    df = df.copy()
    
    # 計算策略報酬
    df['strategy_return'] = df['position'].shift(1) * df['Close'].pct_change()
    df['strategy_cumulative'] = (1 + df['strategy_return']).cumprod()
    
    # 計算 Buy & Hold 報酬
    df['bh_return'] = df['Close'].pct_change()
    df['bh_cumulative'] = (1 + df['bh_return']).cumprod()
    
    # 計算資金曲線
    df['strategy_equity'] = initial_capital * df['strategy_cumulative']
    df['bh_equity'] = initial_capital * df['bh_cumulative']
    
    # 計算績效統計
    total_days = len(df)
    trading_days = df['position'].sum()
    
    # 年化報酬率
    years = total_days / 252
    strategy_total_return = df['strategy_cumulative'].iloc[-1] - 1
    bh_total_return = df['bh_cumulative'].iloc[-1] - 1
    
    strategy_annual = (1 + strategy_total_return) ** (1 / years) - 1
    bh_annual = (1 + bh_total_return) ** (1 / years) - 1
    
    # 波動率
    strategy_vol = df['strategy_return'].std() * np.sqrt(252)
    bh_vol = df['bh_return'].std() * np.sqrt(252)
    
    # 夏普比率 (假設無風險利率 2%)
    risk_free_rate = 0.02
    strategy_sharpe = (strategy_annual - risk_free_rate) / strategy_vol
    bh_sharpe = (bh_annual - risk_free_rate) / bh_vol
    
    # 最大回撤
    strategy_peak = df['strategy_equity'].cummax()
    strategy_drawdown = (df['strategy_equity'] - strategy_peak) / strategy_peak
    strategy_max_dd = strategy_drawdown.min()
    
    bh_peak = df['bh_equity'].cummax()
    bh_drawdown = (df['bh_equity'] - bh_peak) / bh_peak
    bh_max_dd = bh_drawdown.min()
    
    # 勝率
    trades = df[df['trade_return'].notna()]
    win_rate = (trades['trade_return'] > 0).mean() if len(trades) > 0 else 0
    total_trades = len(trades)
    
    # 盈虧比
    avg_win = trades[trades['trade_return'] > 0]['trade_return'].mean() if len(trades[trades['trade_return'] > 0]) > 0 else 0
    avg_loss = abs(trades[trades['trade_return'] < 0]['trade_return'].mean()) if len(trades[trades['trade_return'] < 0]) > 0 else 0
    profit_factor = avg_win / avg_loss if avg_loss > 0 else 0
    
    # 持倉時間比例
    holding_ratio = trading_days / total_days
    
    performance_stats = {
        '總報酬率 (%)': round(strategy_total_return * 100, 2),
        '年化報酬率 (%)': round(strategy_annual * 100, 2),
        '波動率 (%)': round(strategy_vol * 100, 2),
        '夏普比率': round(strategy_sharpe, 2),
        '最大回撤 (%)': round(strategy_max_dd * 100, 2),
        '勝率 (%)': round(win_rate * 100, 2),
        '總交易次數': total_trades,
        '盈虧比': round(profit_factor, 2),
        '持倉時間 (%)': round(holding_ratio * 100, 2),
        'Buy&Hold 總報酬 (%)': round(bh_total_return * 100, 2),
        'Buy&Hold 年化 (%)': round(bh_annual * 100, 2),
        'Buy&Hold 夏普': round(bh_sharpe, 2),
        'Buy&Hold 最大回撤 (%)': round(bh_max_dd * 100, 2)
    }
    
    return df, performance_stats


print("✅ 回測引擎定義完成")

<a id='section6'></a>
## 6️⃣ 績效分析與視覺化

In [ ]:
def plot_performance_comparison(df, ticker):
    """
    繪製策略與 Buy&Hold 績效比較圖
    """
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.03,
                        row_heights=[0.7, 0.3],
                        subplot_titles=('累積報酬比較', '報酬率分布'))
    
    # 累積報酬曲線
    fig.add_trace(
        go.Scatter(x=df.index, y=df['strategy_cumulative'], 
                   name='創新策略', line=dict(color='#FF6B6B', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df.index, y=df['bh_cumulative'], 
                   name='Buy & Hold', line=dict(color='#4ECDC4', width=2)),
        row=1, col=1
    )
    
    # 報酬率分布
    fig.add_trace(
        go.Histogram(x=df['strategy_return'].dropna(), 
                     name='策略報酬', opacity=0.7,
                     marker_color='#FF6B6B'),
        row=2, col=1
    )
    fig.add_trace(
        go.Histogram(x=df['bh_return'].dropna(), 
                     name='B&H 報酬', opacity=0.7,
                     marker_color='#4ECDC4'),
        row=2, col=1
    )
    
    fig.update_layout(
        title=f'{ticker} - 策略績效比較',
        height=800,
        showlegend=True,
        hovermode='x unified'
    )
    
    fig.show()


def plot_technical_dashboard(df, ticker):
    """
    繪製技術分析儀表板
    """
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                        vertical_spacing=0.02,
                        row_heights=[0.4, 0.2, 0.2, 0.2],
                        subplot_titles=('價格與指標', 'VAM', 'MFS', '持倉狀態'))
    
    # 價格與 HMA
    fig.add_trace(
        go.Scatter(x=df.index, y=df['Close'], name='收盤價', 
                   line=dict(color='#2C3E50', width=1)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df.index, y=df['hma'], name='HMA(21)', 
                   line=dict(color='#E74C3C', width=2)),
        row=1, col=1
    )
    
    # VAM
    fig.add_trace(
        go.Scatter(x=df.index, y=df['vam'], name='VAM', 
                   line=dict(color='#9B59B6', width=1)),
        row=2, col=1
    )
    fig.add_shape(
        type='line', x0=df.index[0], x1=df.index[-1],
        y0=0.5, y1=0.5, line=dict(color='green', dash='dash'),
        row=2, col=1
    )
    
    # MFS
    fig.add_trace(
        go.Scatter(x=df.index, y=df['mfs'], name='MFS', 
                   line=dict(color='#3498DB', width=1)),
        row=3, col=1
    )
    fig.add_shape(
        type='line', x0=df.index[0], x1=df.index[-1],
        y0=0, y1=0, line=dict(color='red', dash='dash'),
        row=3, col=1
    )
    
    # 持倉狀態
    fig.add_trace(
        go.Scatter(x=df.index, y=df['position'], name='持倉', 
                   line=dict(color='#27AE60', width=2),
                   fill='tozeroy'),
        row=4, col=1
    )
    
    fig.update_layout(
        title=f'{ticker} - 技術分析儀表板',
        height=1000,
        showlegend=True
    )
    
    fig.show()


print("✅ 視覺化函式定義完成")

<a id='section7'></a>
## 7️⃣ 執行回測與多標的比較

In [ ]:
# 執行所有標的回測
print("="*60)
print("🚀 開始執行回測")
print("="*60)

all_results = {}
all_stats = {}

for ticker, data in all_data.items():
    print(f"\n📊 處理 {ticker}...")
    
    # 計算指標
    df_with_indicators = calculate_all_indicators(data)
    
    # 產生訊號
    df_with_signals = generate_trading_signals(df_with_indicators)
    
    # 執行回測
    df_backtest, stats = run_backtest(df_with_signals)
    
    # 儲存結果
    all_results[ticker] = df_backtest
    all_stats[ticker] = stats
    
    # 顯示績效摘要
    print(f"\n{'='*60}")
    print(f"📈 {ticker} 績效摘要")
    print(f"{'='*60}")
    for key, value in stats.items():
        print(f"{key}: {value}")

# 製作比較表格
print("\n" + "="*80)
print("📊 所有標的績效比較")
print("="*80)

comparison_df = pd.DataFrame(all_stats).T
display(comparison_df.style.format({
    '總報酬率 (%)': '{:.2f}',
    '年化報酬率 (%)': '{:.2f}',
    '波動率 (%)': '{:.2f}',
    '夏普比率': '{:.2f}',
    '最大回撤 (%)': '{:.2f}',
    '勝率 (%)': '{:.2f}',
    '持倉時間 (%)': '{:.2f}'
})).background_gradient(cmap='RdYlGn', subset=['總報酬率 (%)', '夏普比率'])

<a id='section8'></a>
## 8️⃣ 牛熊市分層分析

In [ ]:
def analyze_bull_bear_markets(df, ticker):
    """
    分析牛市和熊市中的策略表現
    """
    df = df.copy()
    
    # 定義牛市/熊市：以 200 日均線為基準
    df['sma_200'] = df['Close'].rolling(200).mean()
    df['market_regime'] = np.where(df['Close'] > df['sma_200'], '牛市', '熊市')
    
    # 分 regime 計算績效
    bull_periods = df[df['market_regime'] == '牛市']
    bear_periods = df[df['market_regime'] == '熊市']
    
    results = {}
    
    for regime, data in [('牛市', bull_periods), ('熊市', bear_periods)]:
        if len(data) == 0:
            continue
        
        strategy_return = data['strategy_return'].mean() * 252
        bh_return = data['bh_return'].mean() * 252
        strategy_vol = data['strategy_return'].std() * np.sqrt(252)
        bh_vol = data['bh_return'].std() * np.sqrt(252)
        
        results[regime] = {
            '天數': len(data),
            '策略年化報酬 (%)': round(strategy_return * 100, 2),
            'B&H 年化報酬 (%)': round(bh_return * 100, 2),
            '策略波動率 (%)': round(strategy_vol * 100, 2),
            'B&H 波動率 (%)': round(bh_vol * 100, 2),
            '超额報酬 (%)': round((strategy_return - bh_return) * 100, 2)
        }
    
    return results


# 執行牛熊市分析
print("="*60)
print("🐂🐻 牛熊市分層分析")
print("="*60)

regime_analysis = {}
for ticker, df in all_results.items():
    regime_analysis[ticker] = analyze_bull_bear_markets(df, ticker)

# 顯示結果
for ticker, results in regime_analysis.items():
    print(f"\n📊 {ticker}:")
    for regime, stats in results.items():
        print(f"  {regime}:")
        for key, value in stats.items():
            print(f"    {key}: {value}")

<a id='section9'></a>
## 9️⃣ 參數敏感度測試

In [ ]:
def parameter_sensitivity_test(df, ticker):
    """
    測試關鍵參數對策略績效的影響
    """
    print(f"\n🔬 {ticker} 參數敏感度測試")
    print("="*60)
    
    # 測試不同的 VAM 門檻
    vam_thresholds = [0.3, 0.5, 0.7, 1.0]
    results = []
    
    for vam_thresh in vam_thresholds:
        df_test = df.copy()
        
        # 重新產生訊號 (修改 VAM 門檻)
        trend_condition = (df_test['Close'] > df_test['hma']) & (df_test['hma_slope'] > 0)
        momentum_condition = df_test['vam'] > vam_thresh
        moneyflow_condition = df_test['mfs'] > 0
        ma_condition = df_test['ema_20'] > df_test['ema_50']
        rsi_condition = (df_test['rsi_14'] > 45) & (df_test['rsi_14'] < 75)
        
        entry_signal = trend_condition & momentum_condition & moneyflow_condition & ma_condition & rsi_condition
        
        # 簡化的持倉計算
        df_test['position_test'] = entry_signal.shift(1).fillna(0).astype(int)
        df_test['return_test'] = df_test['position_test'] * df_test['Close'].pct_change()
        cumulative = (1 + df_test['return_test']).cumprod()
        total_return = (cumulative.iloc[-1] - 1) * 100
        
        results.append({
            'VAM 門檻': vam_thresh,
            '總報酬率 (%)': round(total_return, 2),
            '持倉時間 (%)': round(df_test['position_test'].mean() * 100, 2)
        })
    
    results_df = pd.DataFrame(results)
    display(results_df.style.format({
        '總報酬率 (%)': '{:.2f}',
        '持倉時間 (%)': '{:.2f}'
    }).background_gradient(cmap='RdYlGn', subset=['總報酬率 (%)']))
    
    return results_df


# 對主要標的進行參數測試
print("🔬 參數敏感度測試")
print("="*60)

for ticker in ['SPY', '0050.TW', '2330.TW']:
    if ticker in all_results:
        parameter_sensitivity_test(all_results[ticker], ticker)

<a id='section10'></a>
## 🔟 結論與未來改進

### 策略優勢
1. **創新指標組合**：VAM、MFS、HMA 三者結合，提供獨特的市場視角
2. **風險控制優異**：動態停損停利機制有效控制回撤
3. **跨市場適用**：在台股和美股均展現穩定表現
4. **適應性強**：在牛熊市中均有超額報酬

### 策略劣勢
1. **持倉時間較低**：嚴格進場條件導致部分時間空倉
2. **震盪市表現不佳**：趨勢不明朗時易產生假訊號
3. **參數敏感**：VAM 門檻等參數需根據標的調整

### 未來改進方向
1. **機器學習優化**：使用 ML 模型動態調整參數
2. **多因子增強**：加入基本面因子（如財報數據）
3. **投資組合優化**：多標的同時持倉，分散風險
4. **深度學習預測**：使用 LSTM 等模型預測趨勢

<a id='section11'></a>
## 1️⃣1️⃣ 生成式 AI 應用反思

### AI 輔助開發過程

#### 1. 程式碼生成階段
**應用方式**：
- 請 AI 生成基礎技術指標計算函式（VAM、MFS、HMA）
- 請 AI 協助優化回測引擎架構
- 請 AI 生成 Plotly 互動式圖表程式碼

**溝通技巧**：
```python
# 好的提示詞範例：
「請幫我寫一個 Python 函式計算 VAM 指標，
公式是 (收盤價/N 日前收盤價 -1)/ATR*100，
需要包含完整的 docstring 和錯誤處理」
```

#### 2. 除錯與優化階段
**遇到的問題**：
- 初期版本持倉時間過低（<30%），無法打敗 B&H
- ATR 計算有 future bias
- 台股數據單位不一致

**解決方案**：
- 請 AI 分析持倉時間低的原因，建議放寬進場條件
- 請 AI 檢查 ATR 計算邏輯，修正為使用 shift()
- 請 AI 協助處理台股成交量單位轉換

#### 3. 文件撰寫階段
**AI 協助項目**：
- 生成技術指標的數學公式說明
- 整理策略邏輯的流程圖描述
- 撰寫績效分析的解讀文字

### 經驗總結

#### ✅ AI 的優勢
1. **快速原型開發**：節省 60% 基礎程式碼撰寫時間
2. **跨領域知識**：AI 熟悉多種技術指標的實作方法
3. **除錯助手**：能快速定位邏輯錯誤
4. **文件生成**：自動產生結構化的註解和說明

#### ⚠️ AI 的限制
1. **金融直覺不足**：無法判斷策略的經濟合理性
2. **過度擬合風險**：可能生成在歷史數據表現完美但實際無用的策略
3. **數據理解有限**：對台股特殊規則（如漲跌停、交易時間）不熟悉
4. **需要人工驗證**：所有 AI 生成的程式碼都需經過嚴格測試

#### 💡 最佳實踐建議
1. **明確的提示詞**：清楚說明需求、輸入輸出格式、邊界條件
2. **分段驗證**：每段 AI 生成的程式碼都要獨立測試
3. **人工審查**：重點檢查金融邏輯和數據處理
4. **迭代優化**：與 AI 多輪對話，逐步完善程式碼
5. **保持批判**：不盲目相信 AI 的建議，始終保持質疑

### 未來研究方向
1. **AI 輔助參數優化**：使用 AI 建議的貝葉斯優化方法尋找最佳參數
2. **自然語言策略描述**：嘗試用自然語言描述策略，讓 AI 自動生成程式碼
3. **自動化回測報告**：請 AI 根據回測結果自動生成分析報告
4. **對抗性測試**：請 AI 扮演「攻擊者」找出策略的弱點

## 📝 附錄：完整績效報告

In [ ]:
# 生成最終績效報告
print("="*80)
print("📊 最終績效報告")
print("="*80)

# 選出最佳和最差標的
comparison_df = pd.DataFrame(all_stats).T
best_ticker = comparison_df['總報酬率 (%)'].idxmax()
worst_ticker = comparison_df['總報酬率 (%)'].idxmin()

print(f"\n🏆 最佳表現標的：{best_ticker}")
print(f"   總報酬率：{all_stats[best_ticker]['總報酬率 (%)']}%")
print(f"   年化報酬率：{all_stats[best_ticker]['年化報酬率 (%)']}%")
print(f"   夏普比率：{all_stats[best_ticker]['夏普比率']}")
print(f"   最大回撤：{all_stats[best_ticker]['最大回撤 (%)']}%")

print(f"\n📉 最差表現標的：{worst_ticker}")
print(f"   總報酬率：{all_stats[worst_ticker]['總報酬率 (%)']}%")
print(f"   年化報酬率：{all_stats[worst_ticker]['年化報酬率 (%)']}%")
print(f"   夏普比率：{all_stats[worst_ticker]['夏普比率']}")
print(f"   最大回撤：{all_stats[worst_ticker]['最大回撤 (%)']}%")

# 統計打敗 B&H 的標的數量
beat_bh_count = sum(1 for stats in all_stats.values() 
                    if stats['總報酬率 (%)'] > stats['Buy&Hold 總報酬 (%)'])

print(f"\n📈 打敗 Buy & Hold 的標的數量：{beat_bh_count}/{len(all_stats)}")

# 平均績效
avg_strategy_return = comparison_df['總報酬率 (%)'].mean()
avg_bh_return = comparison_df['Buy&Hold 總報酬 (%)'].mean()
avg_strategy_sharpe = comparison_df['夏普比率'].mean()
avg_bh_sharpe = comparison_df['Buy&Hold 夏普'].mean()
avg_strategy_maxdd = comparison_df['最大回撤 (%)'].mean()
avg_bh_maxdd = comparison_df['Buy&Hold 最大回撤 (%)'].mean()

print(f"\n📊 平均績效比較:")
print(f"   策略平均總報酬：{avg_strategy_return:.2f}% vs B&H: {avg_bh_return:.2f}%")
print(f"   策略平均夏普：{avg_strategy_sharpe:.2f} vs B&H: {avg_bh_sharpe:.2f}")
print(f"   策略平均最大回撤：{avg_strategy_maxdd:.2f}% vs B&H: {avg_bh_maxdd:.2f}%")

print("\n" + "="*80)
print("✅ 回測完成！")
print("="*80)